In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetDeltaWorklist
# MAGIC Publishes current-run Delta work across all eligible connection-owned
# MAGIC queue rows. It performs metadata reads only and never creates adapters or
# MAGIC accesses source secrets.

# COMMAND ----------

# MAGIC %run ../shared/_common

# COMMAND ----------

import json

from pyspark.sql import functions as F

try:
    from src.identifiers import quote_databricks
    from src.worklist_utils import (
        TASK_VALUE_LIMIT_BYTES,
        validate_task_value_payload,
    )
except ModuleNotFoundError:
    from identifiers import quote_databricks
    from worklist_utils import (
        TASK_VALUE_LIMIT_BYTES,
        validate_task_value_payload,
    )

# COMMAND ----------

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog", "da_accelerators")
dbutils.widgets.text("control_schema", "control")
dbutils.widgets.text("max_tables", "0")
dbutils.widgets.text("only_connection_ids", "")
dbutils.widgets.text("only_source_table_ids", "")

run_id = dbutils.widgets.get("run_id").strip() or get_run_id()
catalog = dbutils.widgets.get("catalog").strip() or CATALOG
control_schema = (
    dbutils.widgets.get("control_schema").strip() or CONTROL_SCHEMA
)

only_connection_ids = [
    value.strip()
    for value in dbutils.widgets.get(
        "only_connection_ids"
    ).split(",")
    if value.strip()
]

only_source_table_ids = [
    value.strip()
    for value in dbutils.widgets.get(
        "only_source_table_ids"
    ).split(",")
    if value.strip()
]

for name, value in (
    ("run_id", run_id),
    ("catalog", catalog),
    ("control_schema", control_schema),
):
    if not value:
        raise ValueError(f"{name} is required")

try:
    max_tables = int(
        dbutils.widgets.get("max_tables").strip() or "0"
    )
except ValueError as exc:
    raise ValueError(
        "max_tables must be a non-negative integer"
    ) from exc

if max_tables < 0:
    raise ValueError(
        "max_tables must be a non-negative integer"
    )

if len(only_connection_ids) != len(set(only_connection_ids)):
    raise ValueError(
        "only_connection_ids contains duplicate values"
    )

if len(only_source_table_ids) != len(
    set(only_source_table_ids)
):
    raise ValueError(
        "only_source_table_ids contains duplicate values"
    )

# COMMAND ----------

def ctrl(table_name):
    return (
        f"{quote_databricks(catalog)}."
        f"{quote_databricks(control_schema)}."
        f"{quote_databricks(table_name)}"
    )


queue = spark.table(
    ctrl("delta_sync_queue").replace("`", "")
).alias("q")

control = spark.table(
    ctrl("source_table_control").replace("`", "")
).alias("c")

connections = spark.table(
    ctrl("source_connection").replace("`", "")
).alias("sc")

# COMMAND ----------

eligible = (
    queue.join(
        control,
        (
            F.col("q.connection_id")
            == F.col("c.connection_id")
        )
        & (
            F.col("q.source_table_id")
            == F.col("c.source_table_id")
        ),
        "inner",
    )
    .join(
        connections,
        F.col("c.connection_id")
        == F.col("sc.connection_id"),
        "inner",
    )
    .filter(F.col("q.run_id") == F.lit(run_id))
    .filter(
        F.upper(F.trim(F.col("q.status")))
        == F.lit("QUEUED")
    )
    .filter(
        F.col("q.connection_id").isNotNull()
        & (F.trim(F.col("q.connection_id")) != "")
    )
    .filter(
        F.col("q.source_table_id").isNotNull()
        & (F.trim(F.col("q.source_table_id")) != "")
    )
    .filter(F.col("c.is_active") == F.lit(True))
    .filter(
        F.coalesce(
            F.col("c.initial_load_completed"),
            F.lit(False),
        )
        == F.lit(True)
    )
    .filter(
        F.upper(F.trim(F.col("c.table_decision")))
        == F.lit("AUTO_MIGRATE")
    )
    .filter(F.col("sc.is_active") == F.lit(True))
    .filter(
        F.upper(F.trim(F.col("sc.connection_status")))
        == F.lit("VALID")
    )
    .filter(
        F.col("sc.secret_scope").isNotNull()
        & (F.trim(F.col("sc.secret_scope")) != "")
    )
    .filter(
        F.lower(F.trim(F.col("q.source_system")))
        == F.lower(F.trim(F.col("c.source_system")))
    )
    .filter(
        F.lower(F.trim(F.col("c.source_system")))
        == F.lower(F.trim(F.col("sc.source_system")))
    )
)

if "source_identity_version" in control.columns:
    eligible = eligible.filter(
        F.col("c.source_identity_version") == F.lit(2)
    )

if only_connection_ids:
    eligible = eligible.filter(
        F.col("q.connection_id").isin(
            only_connection_ids
        )
    )

if only_source_table_ids:
    eligible = eligible.filter(
        F.col("q.source_table_id").isin(
            only_source_table_ids
        )
    )

# COMMAND ----------

eligible = (
    eligible.select(
        F.col("q.run_id").alias("run_id"),
        F.col("q.connection_id").alias(
            "connection_id"
        ),
        F.col("q.source_table_id").alias(
            "source_table_id"
        ),
        F.col("c.source_schema").alias(
            "_source_schema"
        ),
        F.col("c.source_table").alias(
            "_source_table"
        ),
    )
    .orderBy(
        "connection_id",
        "_source_schema",
        "_source_table",
        "source_table_id",
    )
)

if max_tables > 0:
    eligible = eligible.limit(max_tables)

# COMMAND ----------

rows = eligible.collect()

worklist = [
    {
        "run_id": row["run_id"],
        "connection_id": row["connection_id"],
        "source_table_id": row["source_table_id"],
    }
    for row in rows
]

ownership_keys = {
    (
        item["run_id"],
        item["connection_id"],
        item["source_table_id"],
    )
    for item in worklist
}

if len(ownership_keys) != len(worklist):
    raise ValueError(
        "Delta worklist contains duplicate ownership keys"
    )

connection_count = len(
    {
        item["connection_id"]
        for item in worklist
    }
)

validate_task_value_payload(
    worklist,
    key="worklist",
    limit_bytes=TASK_VALUE_LIMIT_BYTES,
)

# COMMAND ----------

dbutils.jobs.taskValues.set(
    key="run_id",
    value=run_id,
)

dbutils.jobs.taskValues.set(
    key="worklist",
    value=worklist,
)

dbutils.jobs.taskValues.set(
    key="worklist_count",
    value=len(worklist),
)

dbutils.jobs.taskValues.set(
    key="connection_count",
    value=connection_count,
)

print(
    f"Delta worklist: tables={len(worklist)}, "
    f"connections={connection_count}"
)

# COMMAND ----------

dbutils.notebook.exit(
    json.dumps(
        {
            "status": "SUCCEEDED",
            "run_id": run_id,
            "worklist_count": len(worklist),
            "connection_count": connection_count,
            "worklist": worklist,
        }
    )
)